# theta_interpretation: scalar 1D Gaussian MGD experiment

Goal: fit MGD/SDE maximum-entropy potentials directly on **scalar** 1D
Gaussian data, `x1 ~ N(0, data_sigma^2)`, and look at what the fitted
`theta_t` does under:

1. A **correctly-specified** potential set, `terms=['x2']` — the sufficient
   statistic of a Gaussian.
2. A **misspecified** potential set, `terms=['x2', 'x4']` — adding a term
   the true distribution doesn't need, to see whether the fit correctly
   drives `theta_x4` toward 0 while leaving `theta_x2` essentially unchanged.

**Data layout note:** this is genuinely scalar data, shape `(n1, 1)` — NOT
`(n1, 1, 1)`. `codes/sde_routines.py`'s `SDE` class dispatches on
`len(x_1.shape)`: `2 -> (B, C)` scalar (no wavelet machinery), `3 -> (B, C, T)`
1D field (wavelet scattering over the last axis). A single real number per
sample has no signal-length axis to convolve over, so this experiment uses
the scalar branch and `get_scalar_potentials()` (pointwise `Monomial`
potentials) from `codes/potentials_builder.py`, not the wavelet-scattering
`get_1d_potentials()` the other `run_SDE.py` entry points (jets/turbulence/
gaussian_experiment) use — those work in a genuinely higher-dimensional
`(n_samples, 1, M)` signal space, e.g. `M=128`, which is what "the usual
`(128, 1, 1000)`-shaped data" actually means: `128` samples of a
1000-dimensional field, not a 128-dimensional space sampled 1000 times.

This notebook imports and reuses the functions defined in `run_SDE.py`
(this folder) rather than redefining them, so the notebook and the CLI
script can never drift apart on the run/save/naming logic.


In [ ]:
import sys
from pathlib import Path
from types import SimpleNamespace

import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

root = Path.cwd()
if not (root / 'run_SDE.py').exists():
    root = root / 'theta_interpretation'  # allow running the notebook from the repo root
sys.path.insert(0, str(root))
sys.path.insert(0, str(root.parent / 'codes'))

from run_SDE import (
    make_args,
    run_and_diagnose,
    get_scalar_potentials,
    plot_theta_reg_overlay,
    device,
)
from utils_entropy import entropy_bound, log_Z_bound  # codes/utils_entropy.py, on sys.path via codes/

print('device:', device)


## Expected `theta_x2` (reference — not yet re-verified against this codebase's current SDE derivation)

`codes/sde_routines.py` fits a maximum-entropy density of the form
`p(x) ∝ exp( sum_i theta_i * phi_i(x) )`. This sign/scale convention is
taken from the legacy `notebooks/mgd_sampling/mgd_scalar_example.ipynb`,
which printed a `Target theta` matching it for a quartic target — it has
**not** been independently re-derived against the current `SDE`/
`forward_regularised` implementation, so treat it as a hypothesis to check
against the actual runs below, not a verified fact.

For `phi(x) = x^2` and target `N(0, sigma_data^2)`:

```
p(x) ∝ exp(-x^2 / (2 sigma_data^2))   =>   theta_x2 = -1 / (2 sigma_data^2)
```

For the misspecified set `phi = (x^2, x^4)`, since the true density needs
no quartic term, the expectation is `theta_x2` essentially unchanged and
`theta_x4 -> 0`.


In [ ]:
# make_args()/run_and_diagnose() are imported from run_SDE.py above --
# defined there (not here) so this notebook and the CLI script share the
# exact same run/save/naming logic.

## Run 1: correctly-specified potential, `terms=['x2']`


In [ ]:
run_correct = run_and_diagnose(make_args(['x2'], outdir=str(root)))

In [ ]:
display(Image(filename=str(run_correct['fig_dir'] / 'marginal_histogram.png')))
display(Image(filename=str(run_correct['fig_dir'] / 'theta_trajectory.png')))
display(Image(filename=str(run_correct['fig_dir'] / 'moment_matching.png')))


### Diagnostics: moment matching, entropy bound, regularised vs. raw theta

- **Moment matching** (`moment_matching.png`, already produced by
  `save_diagnostics` in `run_SDE.py`): relative error between the
  interpolant's target moments and the walkers' empirical moments over
  time. If this blows up or never drops below `moment_threshold`, the run
  didn't converge -- don't trust anything else below.
- **Entropy bound check**: `codes/utils_entropy.py`'s `log_Z_bound` (MGD
  Eq. 8) against the *exact* analytic Gaussian `log Z`, which we know here
  because the data-generating process really is `N(0, data_sigma^2)`.
  Also surfaces `entropy_bound`'s own quadrature diagnostics
  (`quadrature_gap`, `refinement`) and `log_Z_bound`'s `m1_discrepancy` --
  use these to judge whether `nt` / `n_subsample` / `lam` are well-tuned
  *before* spending compute on a multi-seed sweep.
- **`theta_t` vs `Theta_reg` overlay**: the raw per-step MGD fit against
  the time-regularised solve, to see what the regularisation is actually
  doing to the trajectory.


In [ ]:
potentials_correct = get_scalar_potentials(run_correct['args'].terms)
term_names_correct = list(potentials_correct.keys())
results_correct = {'run_correct': run_correct['result']}

eb_reg = log_Z_bound(results_correct, 'run_correct', 'Theta_reg', potentials_correct, device=device)
eb_raw = log_Z_bound(results_correct, 'run_correct', 'theta_t', potentials_correct, device=device)

log_Z_analytic = 0.5 * np.log(2 * np.pi * run_correct['args'].data_sigma**2)

print('log_Z_bound (Theta_reg) :', eb_reg['log_Z_bound'])
print('log_Z_bound (theta_t)   :', eb_raw['log_Z_bound'])
print('log_Z_analytic          :', log_Z_analytic)
print()
print('H_bound          :', eb_reg['entropy_bound']['H_bound'])
print('quadrature_gap   :', eb_reg['entropy_bound']['quadrature_gap'],
      '  (trapezoid - left_riemann; large relative to H_bound => nt may be too coarse)')
print('refinement drift :', eb_reg['entropy_bound']['refinement'])
print()
print('m1_discrepancy (Theta_reg)      :', eb_reg['m1_discrepancy'],
      '  (||target - particle moments|| / ||target||; large => corrector/n_subsample/lam not converged)')
print('theta1.m1 bootstrap std (Theta_reg):', eb_reg['theta1_dot_m1_bootstrap_std'])


In [ ]:
plot_theta_reg_overlay(
    run_correct['result']['theta_t'], run_correct['result']['Theta_reg'],
    run_correct['result']['t'], term_names_correct,
    run_correct['fig_dir'], run_correct['config'],
)
display(Image(filename=str(run_correct['fig_dir'] / 'theta_reg_overlay.png')))


In [ ]:
term_names_correct = list(get_scalar_potentials(run_correct['args'].terms).keys())
theta_final_correct = run_correct['result']['theta_t'][-1].detach().cpu()
target_theta_x2 = -1.0 / (2 * run_correct['args'].data_sigma**2)

print('terms:', term_names_correct)
print('fitted theta[-1]:', theta_final_correct.tolist())
print('reference target theta_x2 (see derivation above, unverified):', target_theta_x2)


## Run 2: misspecified potential, `terms=['x2', 'x4']`


In [ ]:
run_wrong = run_and_diagnose(make_args(['x2', 'x4'], outdir=str(root)))

In [ ]:
display(Image(filename=str(run_wrong['fig_dir'] / 'marginal_histogram.png')))
display(Image(filename=str(run_wrong['fig_dir'] / 'theta_trajectory.png')))
display(Image(filename=str(run_wrong['fig_dir'] / 'moment_matching.png')))


### Diagnostics: moment matching, entropy bound, regularised vs. raw theta

- **Moment matching** (`moment_matching.png`, already produced by
  `save_diagnostics` in `run_SDE.py`): relative error between the
  interpolant's target moments and the walkers' empirical moments over
  time. If this blows up or never drops below `moment_threshold`, the run
  didn't converge -- don't trust anything else below.
- **Entropy bound check**: `codes/utils_entropy.py`'s `log_Z_bound` (MGD
  Eq. 8) against the *exact* analytic Gaussian `log Z`, which we know here
  because the data-generating process really is `N(0, data_sigma^2)`.
  Also surfaces `entropy_bound`'s own quadrature diagnostics
  (`quadrature_gap`, `refinement`) and `log_Z_bound`'s `m1_discrepancy` --
  use these to judge whether `nt` / `n_subsample` / `lam` are well-tuned
  *before* spending compute on a multi-seed sweep.
- **`theta_t` vs `Theta_reg` overlay**: the raw per-step MGD fit against
  the time-regularised solve, to see what the regularisation is actually
  doing to the trajectory.


In [ ]:
potentials_wrong = get_scalar_potentials(run_wrong['args'].terms)
term_names_wrong = list(potentials_wrong.keys())
results_wrong = {'run_wrong': run_wrong['result']}

eb_reg = log_Z_bound(results_wrong, 'run_wrong', 'Theta_reg', potentials_wrong, device=device)
eb_raw = log_Z_bound(results_wrong, 'run_wrong', 'theta_t', potentials_wrong, device=device)

log_Z_analytic = 0.5 * np.log(2 * np.pi * run_wrong['args'].data_sigma**2)

print('log_Z_bound (Theta_reg) :', eb_reg['log_Z_bound'])
print('log_Z_bound (theta_t)   :', eb_raw['log_Z_bound'])
print('log_Z_analytic          :', log_Z_analytic)
print()
print('H_bound          :', eb_reg['entropy_bound']['H_bound'])
print('quadrature_gap   :', eb_reg['entropy_bound']['quadrature_gap'],
      '  (trapezoid - left_riemann; large relative to H_bound => nt may be too coarse)')
print('refinement drift :', eb_reg['entropy_bound']['refinement'])
print()
print('m1_discrepancy (Theta_reg)      :', eb_reg['m1_discrepancy'],
      '  (||target - particle moments|| / ||target||; large => corrector/n_subsample/lam not converged)')
print('theta1.m1 bootstrap std (Theta_reg):', eb_reg['theta1_dot_m1_bootstrap_std'])


In [ ]:
plot_theta_reg_overlay(
    run_wrong['result']['theta_t'], run_wrong['result']['Theta_reg'],
    run_wrong['result']['t'], term_names_wrong,
    run_wrong['fig_dir'], run_wrong['config'],
)
display(Image(filename=str(run_wrong['fig_dir'] / 'theta_reg_overlay.png')))


In [ ]:
term_names_wrong = list(get_scalar_potentials(run_wrong['args'].terms).keys())
theta_final_wrong = run_wrong['result']['theta_t'][-1].detach().cpu()

print('terms:', term_names_wrong)
print('fitted theta[-1]:', theta_final_wrong.tolist())


## Side-by-side comparison

If the fit is well-behaved, `theta_x2` should land close to the same value
in both runs (the reference computed above), and the misspecified run's
`theta_x4` should be small relative to `theta_x2` — the model correctly
"discovering" that the quartic term isn't needed.


In [ ]:
i2_correct = term_names_correct.index('x2')
i2_wrong = term_names_wrong.index('x2')
i4_wrong = term_names_wrong.index('x4')

print(f"theta_x2, correct model      : {theta_final_correct[i2_correct].item(): .6f}")
print(f"theta_x2, misspecified model : {theta_final_wrong[i2_wrong].item(): .6f}")
print(f"theta_x4, misspecified model : {theta_final_wrong[i4_wrong].item(): .6f}  (expected near 0)")
print(f"reference target theta_x2    : {target_theta_x2: .6f}")
